# BLAST-Enhanced U-Net Segmentation for Oral Cancer Histopathology

**Objective:** Compare a U-Net model **with vs without BLAST** enhancement for histopathological image segmentation.

| Mode | Description |
|------|-------------|
| **Without BLAST** | U-Net trained on RGB images (10 epochs) |
| **With BLAST** | U-Net trained on 4-channel input: RGB + BLAST prediction map (10 epochs) |

**Classes:** Normal (0), OSCC (1)

**Dataset:** 50 H&E-stained oral histopathology images (25 Normal + 25 OSCC) at 100x & 400x magnification, downloaded from Google Drive. 80/20 stratified train/test split.

**Platform:** Runs on Google Colab or Kaggle with GPU.

In [ ]:
# ── Install Dependencies & Setup ──
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'gdown'])

import os, warnings, time, gc, zipfile
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# Detect platform
ON_COLAB  = os.path.exists('/content')
ON_KAGGLE = os.path.exists('/kaggle')

# GPU setup (platform-aware)
if ON_COLAB:
    nvidia_base = "/usr/local/lib/python3.10/dist-packages/nvidia"
    nvidia_libs = [
        f"{nvidia_base}/cuda_runtime/lib", f"{nvidia_base}/cudnn/lib",
        f"{nvidia_base}/cublas/lib", f"{nvidia_base}/cufft/lib",
        f"{nvidia_base}/curand/lib", f"{nvidia_base}/cusolver/lib",
        f"{nvidia_base}/cusparse/lib", f"{nvidia_base}/nvjitlink/lib",
    ]
    os.environ["LD_LIBRARY_PATH"] = ":".join(nvidia_libs) + ":" + os.environ.get("LD_LIBRARY_PATH", "")

if ON_KAGGLE:
    os.chdir('/kaggle/working')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cv2
import gdown

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, jaccard_score
)
from sklearn.preprocessing import normalize

import tensorflow as tf
from tensorflow.keras import layers, Model

gpus = tf.config.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

platform = 'Colab' if ON_COLAB else ('Kaggle' if ON_KAGGLE else 'Local')
print(f'Platform: {platform}  |  TensorFlow: {tf.__version__}  |  GPUs: {gpus}')

In [ ]:
# ── Configuration ──
SEED       = 42
IMG_SIZE   = 256
PATCH_SIZE = 64
NUM_IMAGES = 50
NUM_CLASSES = 2
EPOCHS     = 10
TOP_K      = 5
TEST_SPLIT = 0.2   # 80% train, 20% test

PATCHES_PER_SIDE  = IMG_SIZE // PATCH_SIZE   # 4
PATCHES_PER_IMAGE = PATCHES_PER_SIDE ** 2    # 16

CLASS_NAMES  = ['Normal', 'OSCC']
CLASS_COLORS = {0: [144, 238, 144], 1: [255, 99, 71]}

np.random.seed(SEED)
tf.random.set_seed(SEED)

OUTPUT_DIR     = 'outputs'
DRIVE_FILE_ID  = '1KvdfS_-7kKSVSoE3JTlSmUX0TBjEJEgb'
ZIP_NAME       = 'oral_cancer_histopath_50.zip'
DATA_DIR       = 'oral_cancer_histopath_50'

os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'Config: {IMG_SIZE}x{IMG_SIZE} images, {PATCH_SIZE}x{PATCH_SIZE} patches, '
      f'{NUM_IMAGES} images, {NUM_CLASSES} classes, {EPOCHS} epochs, '
      f'{int((1-TEST_SPLIT)*100)}/{int(TEST_SPLIT*100)} train/test split')

## Load Real Dataset from Google Drive
Download and extract 50 H&E-stained oral histopathology images (25 Normal + 25 OSCC) from Google Drive. Create uniform segmentation masks from folder-based class labels.

In [ ]:
# ── Download dataset from Google Drive ──
if not os.path.isdir(DATA_DIR):
    print('Downloading dataset from Google Drive...')
    gdown.download(id=DRIVE_FILE_ID, output=ZIP_NAME, quiet=False)
    print('Extracting...')
    with zipfile.ZipFile(ZIP_NAME, 'r') as zf:
        zf.extractall('.')
    os.remove(ZIP_NAME)
    print('Done.')
else:
    print(f'Dataset already exists at {DATA_DIR}/')

# ── Load images and create uniform masks ──
images = []
masks = []
labels = []
filenames = []

for cls_idx, folder in enumerate(['normal', 'oscc']):
    folder_path = os.path.join(DATA_DIR, folder)
    for fname in sorted(os.listdir(folder_path)):
        if not fname.lower().endswith(('.jpeg', '.jpg', '.png')):
            continue
        fpath = os.path.join(folder_path, fname)
        img = cv2.imread(fpath)
        if img is None:
            print(f'  WARNING: could not read {fpath}, skipping')
            continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)

        # Uniform mask: every pixel = class label
        msk = np.full((IMG_SIZE, IMG_SIZE), cls_idx, dtype=np.uint8)

        images.append(img)
        masks.append(msk)
        labels.append(cls_idx)
        filenames.append(f'{folder}/{fname}')

images_np = np.array(images, dtype=np.float32)
masks_np  = np.array(masks, dtype=np.uint8)
labels_np = np.array(labels, dtype=np.int32)

NUM_IMAGES = len(images)

print(f'Loaded {NUM_IMAGES} images: {images_np.shape}, masks: {masks_np.shape}')
print(f'  Normal: {(labels_np == 0).sum()}, OSCC: {(labels_np == 1).sum()}')

In [ ]:
# ── Train / Test Split (stratified) ──
from sklearn.model_selection import train_test_split

indices = np.arange(NUM_IMAGES)
train_idx, test_idx = train_test_split(
    indices, test_size=TEST_SPLIT, random_state=SEED, stratify=labels_np
)

X_train = images_np[train_idx]
y_train = masks_np[train_idx][..., np.newaxis]
X_test  = images_np[test_idx]
y_test  = masks_np[test_idx]

train_labels = labels_np[train_idx]
test_labels  = labels_np[test_idx]

print(f'Train: {len(train_idx)} images  (Normal: {(train_labels==0).sum()}, OSCC: {(train_labels==1).sum()})')
print(f'Test:  {len(test_idx)} images  (Normal: {(test_labels==0).sum()}, OSCC: {(test_labels==1).sum()})')

In [ ]:
# Visualize sample images with ground truth masks
nc_count = (labels_np == 0).sum()
sample_imgs = [0, nc_count - 1, nc_count, NUM_IMAGES - 1]

fig, axes = plt.subplots(len(sample_imgs), 3, figsize=(12, 4 * len(sample_imgs)))
for row, img_id in enumerate(sample_imgs):
    axes[row, 0].imshow(images[img_id])
    axes[row, 0].set_title(f'{filenames[img_id]}', fontsize=9)
    axes[row, 0].axis('off')

    color_mask = np.zeros((*masks[img_id].shape, 3), dtype=np.uint8)
    for c, clr in CLASS_COLORS.items():
        color_mask[masks[img_id] == c] = clr
    axes[row, 1].imshow(color_mask)
    axes[row, 1].set_title(f'GT: {CLASS_NAMES[labels[img_id]]}', fontsize=10)
    axes[row, 1].axis('off')

    overlay = (images[img_id].astype(np.float32) * 0.6 +
               color_mask.astype(np.float32) * 0.4).astype(np.uint8)
    axes[row, 2].imshow(overlay)
    axes[row, 2].set_title('Overlay', fontsize=10)
    axes[row, 2].axis('off')

legend_patches = [mpatches.Patch(color=np.array(c)/255, label=n)
                  for n, c in zip(CLASS_NAMES, CLASS_COLORS.values())]
fig.legend(handles=legend_patches, loc='upper center', ncol=len(CLASS_NAMES), fontsize=12)
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig(os.path.join(OUTPUT_DIR, 'sample_images_masks.png'), dpi=120, bbox_inches='tight')
plt.show()

## U-Net Model
Define a compact U-Net architecture for semantic segmentation.

In [ ]:
def build_unet(input_channels=3, num_classes=NUM_CLASSES):
    """Build a compact U-Net for semantic segmentation."""
    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, input_channels))

    # Encoder
    c1 = layers.Conv2D(32, 3, activation='relu', padding='same')(inputs)
    c1 = layers.Conv2D(32, 3, activation='relu', padding='same')(c1)
    p1 = layers.MaxPooling2D(2)(c1)

    c2 = layers.Conv2D(64, 3, activation='relu', padding='same')(p1)
    c2 = layers.Conv2D(64, 3, activation='relu', padding='same')(c2)
    p2 = layers.MaxPooling2D(2)(c2)

    c3 = layers.Conv2D(128, 3, activation='relu', padding='same')(p2)
    c3 = layers.Conv2D(128, 3, activation='relu', padding='same')(c3)
    p3 = layers.MaxPooling2D(2)(c3)

    # Bottleneck
    c4 = layers.Conv2D(256, 3, activation='relu', padding='same')(p3)
    c4 = layers.Conv2D(256, 3, activation='relu', padding='same')(c4)

    # Decoder
    u5 = layers.UpSampling2D(2)(c4)
    u5 = layers.Concatenate()([u5, c3])
    c5 = layers.Conv2D(128, 3, activation='relu', padding='same')(u5)
    c5 = layers.Conv2D(128, 3, activation='relu', padding='same')(c5)

    u6 = layers.UpSampling2D(2)(c5)
    u6 = layers.Concatenate()([u6, c2])
    c6 = layers.Conv2D(64, 3, activation='relu', padding='same')(u6)
    c6 = layers.Conv2D(64, 3, activation='relu', padding='same')(c6)

    u7 = layers.UpSampling2D(2)(c6)
    u7 = layers.Concatenate()([u7, c1])
    c7 = layers.Conv2D(32, 3, activation='relu', padding='same')(u7)
    c7 = layers.Conv2D(32, 3, activation='relu', padding='same')(c7)

    outputs = layers.Conv2D(num_classes, 1, activation='softmax')(c7)

    model = Model(inputs, outputs)
    model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model


# Show model summary
test_model = build_unet(input_channels=3)
print(f'U-Net (3ch) params: {test_model.count_params():,}')
del test_model

test_model = build_unet(input_channels=4)
print(f'U-Net (4ch) params: {test_model.count_params():,}')
del test_model
tf.keras.backend.clear_session()

In [ ]:
def compute_segmentation_metrics(true_masks, pred_seg_maps, method_name):
    """Compute pixel-level segmentation metrics on test set."""
    all_true, all_pred = [], []
    for img_id in test_idx:
        all_true.append(true_masks[img_id].flatten())
        all_pred.append(pred_seg_maps[img_id].flatten())

    all_true = np.concatenate(all_true)
    all_pred = np.concatenate(all_pred)

    pixel_acc = accuracy_score(all_true, all_pred)
    mean_iou  = jaccard_score(all_true, all_pred, average='macro')

    dice_scores = []
    for c in range(NUM_CLASSES):
        gt_c   = (all_true == c)
        pred_c = (all_pred == c)
        intersection = (gt_c & pred_c).sum()
        dice = 2 * intersection / (gt_c.sum() + pred_c.sum() + 1e-10)
        dice_scores.append(dice)
    mean_dice = np.mean(dice_scores)

    prec = precision_score(all_true, all_pred, average='macro', zero_division=0)
    rec  = recall_score(all_true, all_pred, average='macro', zero_division=0)
    f1   = f1_score(all_true, all_pred, average='macro', zero_division=0)

    return {
        'Method': method_name,
        'Pixel Accuracy': pixel_acc,
        'Mean IoU': mean_iou,
        'Mean Dice': mean_dice,
        'Precision': prec,
        'Recall': rec,
        'F1 Score': f1,
        'Dice_per_class': dice_scores,
        'all_true': all_true,
        'all_pred': all_pred,
    }


print('Metrics function defined.')

## Mode 1: U-Net Without BLAST
Train the U-Net on RGB images for 10 epochs using 80/20 train/test split.

In [ ]:
print('MODE 1: U-Net - Direct Segmentation (No BLAST)')
print('=' * 55)

tf.keras.backend.clear_session()
gc.collect()

t0 = time.time()

model_direct = build_unet(input_channels=3)
history_direct = model_direct.fit(X_train, y_train, epochs=EPOCHS, batch_size=4,
                                   validation_split=0.1, verbose=1)

# Predict on test set
direct_seg_maps = {}
for i, img_id in enumerate(test_idx):
    pred_prob = model_direct.predict(images_np[img_id:img_id+1], verbose=0)[0]
    direct_seg_maps[img_id] = np.argmax(pred_prob, axis=-1).astype(np.uint8)

elapsed = time.time() - t0
m_direct = compute_segmentation_metrics(masks, direct_seg_maps, 'U-Net (Direct)')
print(f'\nDone in {elapsed:.0f}s')
print(f'Acc: {m_direct["Pixel Accuracy"]:.4f}  IoU: {m_direct["Mean IoU"]:.4f}  '
      f'Dice: {m_direct["Mean Dice"]:.4f}  F1: {m_direct["F1 Score"]:.4f}')

del model_direct
tf.keras.backend.clear_session()
gc.collect()

## BLAST Technique
Extract patch-level CNN features, perform k-NN matching against a reference database, and generate a BLAST prediction map to use as a 4th input channel.

In [ ]:
class BLASTImageDatabase:
    """BLAST-like database for histopathological patch matching."""

    def __init__(self, metric='cosine', top_k=TOP_K):
        self.metric = metric
        self.top_k = top_k
        self.db_features = None
        self.db_labels = None

    def build_database(self, features, labels):
        self.db_features = normalize(features.copy())
        self.db_labels = labels.copy()

    def query(self, query_feature):
        query_norm = normalize(query_feature.reshape(1, -1))
        if self.metric == 'cosine':
            sims = (query_norm @ self.db_features.T).flatten()
        else:
            dists = np.linalg.norm(self.db_features - query_norm, axis=1)
            sims = 1.0 / (1.0 + dists)

        top_idx = np.argsort(sims)[::-1][:self.top_k]
        top_labels = self.db_labels[top_idx]
        weights = np.maximum(sims[top_idx], 1e-10)

        class_scores = np.zeros(NUM_CLASSES)
        for lbl, w in zip(top_labels, weights):
            class_scores[lbl] += w

        return np.argmax(class_scores)

    def segment_image(self, patch_features):
        """Query all patches and assemble full-resolution segmentation map."""
        preds = [self.query(f) for f in patch_features]
        pred_grid = np.array(preds).reshape(PATCHES_PER_SIDE, PATCHES_PER_SIDE)
        seg_map = np.kron(pred_grid, np.ones((PATCH_SIZE, PATCH_SIZE), dtype=np.uint8))
        return seg_map


def build_cnn_feature_extractor():
    """Small CNN for patch-level feature extraction (trained as classifier)."""
    inputs = layers.Input(shape=(PATCH_SIZE, PATCH_SIZE, 3))
    x = layers.Conv2D(32, 3, activation='relu', padding='same')(inputs)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Conv2D(64, 3, activation='relu', padding='same')(x)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Conv2D(128, 3, activation='relu', padding='same')(x)
    features = layers.GlobalAveragePooling2D(name='features')(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax')(features)

    classifier = Model(inputs, outputs)
    classifier.compile(optimizer='adam',
                       loss='sparse_categorical_crossentropy',
                       metrics=['accuracy'])

    # Feature extractor shares weights, outputs from 'features' layer
    extractor = Model(inputs, classifier.get_layer('features').output)
    return classifier, extractor


def extract_all_patches(images_list):
    """Extract non-overlapping patches and labels from all images."""
    all_patches, all_labels, img_indices = [], [], []

    for img_id, (img, msk) in enumerate(zip(images_list, masks)):
        for r in range(PATCHES_PER_SIDE):
            for c_idx in range(PATCHES_PER_SIDE):
                y0, x0 = r * PATCH_SIZE, c_idx * PATCH_SIZE
                patch = img[y0:y0+PATCH_SIZE, x0:x0+PATCH_SIZE]
                label_patch = msk[y0:y0+PATCH_SIZE, x0:x0+PATCH_SIZE]
                all_patches.append(patch)
                vals, counts = np.unique(label_patch, return_counts=True)
                all_labels.append(vals[np.argmax(counts)])
                img_indices.append(img_id)

    return np.array(all_patches), np.array(all_labels), np.array(img_indices)


print('BLAST database class and helper functions defined.')

## Mode 2: U-Net With BLAST Enhancement
Build BLAST prediction maps from training patches, concatenate as 4th channel (RGB + BLAST), then train U-Net for 10 epochs using 80/20 split.

In [ ]:
print('MODE 2: U-Net + BLAST Enhancement')
print('=' * 55)

t0 = time.time()

# --- Step 1: Extract patches from ALL images ---
all_patches, all_labels, img_indices = extract_all_patches(images)
patches_float = all_patches.astype(np.float32) / 255.0

train_patch_mask = np.isin(img_indices, train_idx)
test_patch_mask  = np.isin(img_indices, test_idx)

# --- Step 2: Train CNN feature extractor on training patches ---
tf.keras.backend.clear_session()
gc.collect()

classifier, extractor = build_cnn_feature_extractor()
classifier.fit(patches_float[train_patch_mask], all_labels[train_patch_mask],
               epochs=5, batch_size=32, verbose=1)
print('CNN feature extractor trained.')

# --- Step 3: Extract embeddings ---
all_embeddings = extractor.predict(patches_float, batch_size=32, verbose=0)
print(f'Extracted {all_embeddings.shape[0]} patch embeddings (dim={all_embeddings.shape[1]})')

del classifier, extractor
tf.keras.backend.clear_session()
gc.collect()

# --- Step 4: Build BLAST database from training patches ---
db = BLASTImageDatabase(metric='cosine', top_k=TOP_K)
db.build_database(all_embeddings[train_patch_mask], all_labels[train_patch_mask])

# --- Step 5: Generate BLAST maps for train and test images ---
blast_maps = {}
for img_id in list(train_idx) + list(test_idx):
    query_mask = img_indices == img_id
    blast_maps[img_id] = db.segment_image(all_embeddings[query_mask])
print(f'Generated BLAST maps for {len(blast_maps)} images.')

# --- Step 6: Build 4-channel inputs ---
X_train_4ch = np.zeros((len(train_idx), IMG_SIZE, IMG_SIZE, 4), dtype=np.float32)
for k, tr_id in enumerate(train_idx):
    X_train_4ch[k, :, :, :3] = images_np[tr_id]
    X_train_4ch[k, :, :, 3] = blast_maps[tr_id].astype(np.float32) * (255.0 / max(NUM_CLASSES - 1, 1))

y_train_4ch = masks_np[train_idx][..., np.newaxis]

# --- Step 7: Train U-Net on 4-channel input ---
tf.keras.backend.clear_session()
gc.collect()

model_blast = build_unet(input_channels=4)
history_blast = model_blast.fit(X_train_4ch, y_train_4ch, epochs=EPOCHS, batch_size=4,
                                 validation_split=0.1, verbose=1)

# --- Step 8: Predict on test set ---
blast_seg_maps = {}
for img_id in test_idx:
    X_test_4ch = np.zeros((1, IMG_SIZE, IMG_SIZE, 4), dtype=np.float32)
    X_test_4ch[0, :, :, :3] = images_np[img_id]
    X_test_4ch[0, :, :, 3] = blast_maps[img_id].astype(np.float32) * (255.0 / max(NUM_CLASSES - 1, 1))
    pred_prob = model_blast.predict(X_test_4ch, verbose=0)[0]
    blast_seg_maps[img_id] = np.argmax(pred_prob, axis=-1).astype(np.uint8)

elapsed = time.time() - t0
m_blast = compute_segmentation_metrics(masks, blast_seg_maps, 'U-Net + BLAST')
print(f'\nDone in {elapsed:.0f}s')
print(f'Acc: {m_blast["Pixel Accuracy"]:.4f}  IoU: {m_blast["Mean IoU"]:.4f}  '
      f'Dice: {m_blast["Mean Dice"]:.4f}  F1: {m_blast["F1 Score"]:.4f}')

del model_blast, X_train_4ch
tf.keras.backend.clear_session()
gc.collect()

## Results Comparison

In [ ]:
# Comparison table
all_metrics = [m_direct, m_blast]

metrics_df = pd.DataFrame([
    {k: v for k, v in m.items() if k not in ['Dice_per_class', 'all_true', 'all_pred']}
    for m in all_metrics
]).set_index('Method').round(4)

print('=' * 75)
print('U-Net: With BLAST vs Without BLAST')
print('=' * 75)
metrics_df

In [ ]:
# Grouped bar chart
metric_cols = ['Pixel Accuracy', 'Mean IoU', 'Mean Dice', 'Precision', 'Recall', 'F1 Score']

direct_vals = [metrics_df.loc['U-Net (Direct)', c] for c in metric_cols]
blast_vals  = [metrics_df.loc['U-Net + BLAST', c] for c in metric_cols]

x = np.arange(len(metric_cols))
w = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(x - w/2, direct_vals, w, label='U-Net (Direct)', color='#C44E52')
ax.bar(x + w/2, blast_vals,  w, label='U-Net + BLAST',  color='#4C72B0')
ax.set_xticks(x)
ax.set_xticklabels(metric_cols, fontsize=11)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score', fontsize=12)
ax.legend(fontsize=12)
ax.set_title('U-Net: With BLAST vs Without BLAST', fontsize=15)

# Add value labels on bars
for i, (d, b) in enumerate(zip(direct_vals, blast_vals)):
    ax.text(i - w/2, d + 0.02, f'{d:.3f}', ha='center', fontsize=9)
    ax.text(i + w/2, b + 0.02, f'{b:.3f}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'unet_blast_vs_direct.png'), dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Confusion matrices side by side
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))

for ax, m in zip(axes, all_metrics):
    cm = confusion_matrix(m['all_true'], m['all_pred'], labels=list(range(NUM_CLASSES)))
    cm_norm = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-10)

    im = ax.imshow(cm_norm, vmin=0, vmax=1, cmap='Blues')
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            ax.text(j, i, f'{cm_norm[i,j]:.2f}', ha='center', va='center',
                    fontsize=11, color='white' if cm_norm[i,j] > 0.5 else 'black')
    ax.set_xticks(range(NUM_CLASSES))
    ax.set_yticks(range(NUM_CLASSES))
    ax.set_xticklabels(CLASS_NAMES, fontsize=9)
    ax.set_yticklabels(CLASS_NAMES, fontsize=9)
    ax.set_title(m['Method'], fontsize=12)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')

plt.suptitle('Confusion Matrices (Normalized)', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'confusion_matrices.png'), dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Visual segmentation comparison (test images only)
sample_test = list(test_idx[:4])

fig, axes = plt.subplots(len(sample_test), 4, figsize=(16, 4 * len(sample_test)))

for row, img_id in enumerate(sample_test):
    # Original
    axes[row, 0].imshow(images[img_id])
    axes[row, 0].set_title(f'{filenames[img_id]}', fontsize=9)
    axes[row, 0].axis('off')

    # Ground truth
    gt_color = np.zeros((*masks[img_id].shape, 3), dtype=np.uint8)
    for c, clr in CLASS_COLORS.items():
        gt_color[masks[img_id] == c] = clr
    axes[row, 1].imshow(gt_color)
    axes[row, 1].set_title('Ground Truth' if row == 0 else '', fontsize=11)
    axes[row, 1].axis('off')

    # U-Net Direct
    seg_d = direct_seg_maps[img_id]
    seg_d_color = np.zeros((*seg_d.shape, 3), dtype=np.uint8)
    for c, clr in CLASS_COLORS.items():
        seg_d_color[seg_d == c] = clr
    axes[row, 2].imshow(seg_d_color)
    axes[row, 2].set_title('U-Net (Direct)' if row == 0 else '', fontsize=11)
    axes[row, 2].axis('off')

    # U-Net + BLAST
    seg_b = blast_seg_maps[img_id]
    seg_b_color = np.zeros((*seg_b.shape, 3), dtype=np.uint8)
    for c, clr in CLASS_COLORS.items():
        seg_b_color[seg_b == c] = clr
    axes[row, 3].imshow(seg_b_color)
    axes[row, 3].set_title('U-Net + BLAST' if row == 0 else '', fontsize=11)
    axes[row, 3].axis('off')

legend_patches = [mpatches.Patch(color=np.array(c)/255, label=n)
                  for n, c in zip(CLASS_NAMES, CLASS_COLORS.values())]
fig.legend(handles=legend_patches, loc='upper center', ncol=len(CLASS_NAMES), fontsize=12)
plt.suptitle('Segmentation Maps: U-Net Direct vs U-Net + BLAST (Test Set)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'unet_segmentation_maps.png'), dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Save metrics to CSV
metrics_df.to_csv(os.path.join(OUTPUT_DIR, 'segmentation_metrics.csv'))
print(f'Metrics saved to {OUTPUT_DIR}/segmentation_metrics.csv')
metrics_df